In [1]:
!git clone https://github.com/Shankha3/codealpha_tasks.git

Cloning into 'codealpha_tasks'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 467.83 KiB | 3.20 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
%cd /content/codealpha_tasks
from google.colab import drive
drive.mount('/content/drive')

/content/codealpha_tasks
Mounted at /content/drive


In [5]:
!find /content -name "Car_Price_Prediction.ipynb"

/content/drive/MyDrive/Colab Notebooks/Car_Price_Prediction.ipynb


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Unemployment_Analysis.ipynb" .
!ls

In [ ]:
# Task 3: Car Price Prediction with Machine Learning
# Using Regression Models with Pandas, Scikit-learn, Matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─────────────────────────────────────────
# 1. CREATE REALISTIC DATASET
# ─────────────────────────────────────────
np.random.seed(42)
n = 500

brands = ['Toyota', 'Honda', 'BMW', 'Audi', 'Ford',
          'Hyundai', 'Maruti', 'Kia', 'Mercedes', 'Tata']
fuel_types = ['Petrol', 'Diesel', 'Electric', 'CNG']
transmissions = ['Manual', 'Automatic']

brand_goodwill = {
    'Mercedes': 5, 'BMW': 5, 'Audi': 4,
    'Toyota': 4,  'Honda': 3, 'Ford': 3,
    'Hyundai': 3, 'Kia': 3,  'Maruti': 2, 'Tata': 2
}

brand_arr       = np.random.choice(brands, n)
fuel_arr        = np.random.choice(fuel_types, n, p=[0.45,0.30,0.15,0.10])
transmission_arr= np.random.choice(transmissions, n, p=[0.55, 0.45])
year_arr        = np.random.randint(2010, 2024, n)
horsepower_arr  = np.random.randint(70, 400, n)
mileage_arr     = np.random.uniform(8, 35, n).round(2)   # kmpl
engine_cc_arr   = np.random.randint(800, 5000, n)
seats_arr       = np.random.choice([4, 5, 6, 7], n, p=[0.1,0.6,0.1,0.2])
km_driven_arr   = np.random.randint(0, 200000, n)
goodwill_arr    = np.array([brand_goodwill[b] for b in brand_arr])

# Price formula (realistic)
price_arr = (
    goodwill_arr * 120000
    + horsepower_arr * 800
    + (2024 - year_arr) * (-25000)
    + mileage_arr * 5000
    + engine_cc_arr * 10
    - km_driven_arr * 0.5
    + (transmission_arr == 'Automatic') * 80000
    + (fuel_arr == 'Electric') * 200000
    + np.random.normal(0, 50000, n)
).clip(100000, 10000000)

df = pd.DataFrame({
    'Brand'       : brand_arr,
    'Fuel_Type'   : fuel_arr,
    'Transmission': transmission_arr,
    'Year'        : year_arr,
    'Horsepower'  : horsepower_arr,
    'Mileage_kmpl': mileage_arr,
    'Engine_CC'   : engine_cc_arr,
    'Seats'       : seats_arr,
    'KM_Driven'   : km_driven_arr,
    'Brand_Goodwill': goodwill_arr,
    'Price'       : price_arr.astype(int)
})

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nBasic Statistics:")
print(df.describe())

# ─────────────────────────────────────────
# 2. DATA PREPROCESSING
# ─────────────────────────────────────────
print("\nMissing Values:", df.isnull().sum().sum())

# Encode categorical columns
le = LabelEncoder()
df_encoded = df.copy()
for col in ['Brand', 'Fuel_Type', 'Transmission']:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Features & Target
X = df_encoded.drop('Price', axis=1)
y = df_encoded['Price']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\nTraining samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

# ─────────────────────────────────────────
# 3. TRAIN MODELS
# ─────────────────────────────────────────
models = {
    'Linear Regression'   : LinearRegression(),
    'Random Forest'       : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    predictions[name] = y_pred

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

    print(f"\n{'='*45}")
    print(f"Model : {name}")
    print(f"MAE   : ₹{mae:,.0f}")
    print(f"RMSE  : ₹{rmse:,.0f}")
    print(f"R²    : {r2:.4f}")

# ─────────────────────────────────────────
# 4. VISUALIZATIONS
# ─────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(18, 14))

# --- Plot 1: Actual vs Predicted (Best Model) ---
best_name = max(results, key=lambda x: results[x]['R2'])
best_pred = predictions[best_name]

ax1 = fig.add_subplot(3, 2, 1)
ax1.scatter(y_test, best_pred, alpha=0.5, color='steelblue')
ax1.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_title(f'Actual vs Predicted — {best_name}')
ax1.set_xlabel('Actual Price (₹)')
ax1.set_ylabel('Predicted Price (₹)')

# --- Plot 2: Model Comparison (R²) ---
ax2 = fig.add_subplot(3, 2, 2)
names = list(results.keys())
r2_scores = [results[n]['R2'] for n in names]
bars = ax2.bar(names, r2_scores, color=['steelblue','green','orange'],
               edgecolor='black')
ax2.set_title('Model Comparison — R² Score')
ax2.set_ylabel('R² Score')
ax2.set_ylim(0, 1)
for bar, val in zip(bars, r2_scores):
    ax2.text(bar.get_x() + bar.get_width()/2,
             val + 0.01, f'{val:.3f}', ha='center', fontweight='bold')

# --- Plot 3: Feature Importance (Random Forest) ---
ax3 = fig.add_subplot(3, 2, 3)
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_,
                         index=X.columns).sort_values(ascending=True)
importances.plot(kind='barh', ax=ax3, color='teal')
ax3.set_title('Feature Importance — Random Forest')
ax3.set_xlabel('Importance Score')

# --- Plot 4: Price Distribution ---
ax4 = fig.add_subplot(3, 2, 4)
ax4.hist(df['Price'] / 100000, bins=30,
         color='purple', edgecolor='black', alpha=0.7)
ax4.set_title('Car Price Distribution')
ax4.set_xlabel('Price (₹ Lakhs)')
ax4.set_ylabel('Frequency')

# --- Plot 5: Brand vs Avg Price ---
ax5 = fig.add_subplot(3, 2, 5)
brand_avg = df.groupby('Brand')['Price'].mean().sort_values() / 100000
brand_avg.plot(kind='barh', ax=ax5, color='coral', edgecolor='black')
ax5.set_title('Average Price by Brand')
ax5.set_xlabel('Avg Price (₹ Lakhs)')

# --- Plot 6: Correlation Heatmap ---
ax6 = fig.add_subplot(3, 2, 6)
corr = df_encoded.corr()
sns.heatmap(corr[['Price']].sort_values('Price', ascending=False),
            annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax6, linewidths=0.5)
ax6.set_title('Feature Correlation with Price')

plt.suptitle('Car Price Prediction — ML Analysis',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('car_price_prediction.png', bbox_inches='tight', dpi=150)
plt.show()

# ─────────────────────────────────────────
# 5. PREDICT ON NEW CAR
# ─────────────────────────────────────────
print("\n" + "="*50)
print("         PREDICTING PRICE FOR A NEW CAR")
print("="*50)

new_car = pd.DataFrame([{
    'Brand'          : 2,   # Honda encoded
    'Fuel_Type'      : 2,   # Petrol encoded
    'Transmission'   : 0,   # Manual encoded
    'Year'           : 2020,
    'Horsepower'     : 120,
    'Mileage_kmpl'   : 18.5,
    'Engine_CC'      : 1500,
    'Seats'          : 5,
    'KM_Driven'      : 35000,
    'Brand_Goodwill' : 3
}])

new_car_sc = scaler.transform(new_car)
best_model = models[best_name]
predicted_price = best_model.predict(new_car_sc)[0]

print(f"\nCar Details   : Honda, Petrol, Manual, 2020, 120HP")
print(f"Best Model    : {best_name}")
print(f"Predicted Price: ₹{predicted_price:,.0f}  "
      f"(₹{predicted_price/100000:.2f} Lakhs)")

print("\n✅ Task 3 Complete!")